# 06 - MRT Acquisition Pilot

Controlled dry-run only. This notebook selects 9 TRAIN incidents from frozen Roster v2, builds proposed RouteViews UPDATE acquisition windows, and writes pilot manifests/audits. It does not download MRT files, parse BGP data, create Normal labels, generate features, train models, or touch Validation/Holdout incidents.

## Scope Guardrails

- Use TRAIN incidents only.
- Select 3 Hijack, 3 Leak, and 3 Outage incidents.
- Within each class, choose short, median, and long duration examples.
- Use RouteViews only for this pilot unless later review requires RIPE RIS.
- Generate URLs and local destinations, but do not download files.

In [ ]:
from pathlib import Path
import csv
import json
import sys
from collections import Counter

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent.resolve()
sys.path.insert(0, str(ROOT / 'src'))

from bgp_incident_detection.mrt_acquisition import run_dry_run

def read_csv(path):
    with path.open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

## Run Dry-Run Planner

In [ ]:
results = run_dry_run()
pilot = results['pilot']
plan = results['plan']
audit = results['audit']

print('Pilot incidents:', len(pilot))
print('Class counts:', Counter(row['category'] for row in pilot))
print('Partitions:', Counter(row['partition'] for row in pilot))
print('Audit statuses:', Counter(row['status'] for row in audit))

## Pilot Manifest

In [ ]:
read_csv(ROOT / 'data/pilot/mrt_pilot_incidents_v1.csv')

## Acquisition Window Summary

In [ ]:
summary = []
for row in read_csv(ROOT / 'data/pilot/mrt_pilot_acquisition_plan_v1.csv'):
    urls = json.loads(row['mrt_urls'])
    summary.append({
        'incident_uid': row['incident_uid'],
        'category': row['category'],
        'collector': row['collector'],
        'collection_start': row['collection_start'],
        'collection_end': row['collection_end'],
        'expected_file_count': row['expected_file_count'],
        'first_url': urls[0],
        'last_url': urls[-1],
        'local_destination': row['local_destination'],
    })
summary

## RouteViews Source Plan

RouteViews UPDATE files are generated every 15 minutes. The dry-run uses collector `route-views2` and URL pattern `https://archive.routeviews.org/bgpdata/YYYY.MM/UPDATES/updates.YYYYMMDD.HHMM.bz2`. Incident timestamps are UTC; collection starts are floored to the previous 15-minute boundary and collection ends are ceiled to the next 15-minute boundary after adding configurable context buffers.

## Active-Window Preparation Notes

Hijack and Leak rows preserve detector first and last evidence from the frozen roster. The pilot does not assume that the full first-to-last detector span is continuously active. Outage rows preserve the Cloudflare documented `startDate -> endDate` interval.

## Audit

In [ ]:
audit_rows = read_csv(ROOT / 'data/pilot/mrt_pilot_acquisition_audit.csv')
Counter(row['status'] for row in audit_rows), audit_rows